# Ingest circuits.csv file
1. Read the file using spark dataframe reader API
1. Add Metadata Columns 
    - Source File
    - Ingestion Timestamp
1. Write to bronze delta table    

####`Step`-1 read the csv file using the dataframe reader api

In [0]:
%run ../00-common-Config/01-environment-variable

In [0]:
%run ../00-common-Config/02-helper-function

In [0]:
dbutils.widgets.text("p_batch_id","")
v_batch_id=dbutils.widgets.get("p_batch_id")

In [0]:
landing_path
Source_file=f"{landing_path}/{v_batch_id}/circuits.csv"
table_name=f"{catalog_name}.{bronze_schema}.circuits"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

circuits_schema = StructType(
    [StructField("circuitId", StringType(), True),
     StructField("url", StringType(), True),
     StructField("circuitName", StringType(), True),
     StructField("lat", DoubleType(), True),
     StructField("long", DoubleType(), True),
     StructField("locality", StringType(), True),
     StructField("country", StringType(), True)]
)

In [0]:
circuit_df =spark.read.format("csv")\
    .option("header","true")\
    .schema(circuits_schema)\
    .option("mode","FAILFAST")\
    .load(Source_file)

    # .option("inferSchema","true")\
circuit_df.show()

##Adding the metadata column 
sourcefile
ingestionTimeStmap


In [0]:
# from pyspark.sql import functions as F

# circuit_df  = (
#     circuit_df
#     .withColumn("ingestion_timestamp", F.current_timestamp())
#     .withColumn("source_file", F.col('_metadata.file_path'))
#     )
circuit_df=add_ingestion_metadata(circuit_df)
circuit_df.show()

In [0]:
# circuit_final_df=circuit_df.withColumn("batch_id",f.lit(v_batch_id))

### writting the Data into the Bronze table 

In [0]:
# circuit_final_df.write.format("delta")\
#     .mode("overwrite")\
#     .partitionBy("batch_id")\
#     .option("replaceWhere",f"batch_id='{v_batch_id}'")\
#     .saveAsTable(table_name)

In [0]:
write_to_bronze(
    input_df=circuit_df,
    target_table=table_name,
    batch_id=v_batch_id
)

In [0]:
spark.read.table(table_name).show()